In [7]:
import csv
import re

INPUTFOLDER = "../../data/Zulassungszahlen/Rohdaten/"
OUTPUTFOLDER = "../../data/Zulassungszahlen/"
YEARS  = [2021, 2022, 2023, 2024, 2025]
SOURCES = [
    ("FahrzeugArt-fz1", "FZ"),
    ("AntriebsArt-fz1", "AA"),
]

# Metadaten-Spalten, die nicht übernommen werden sollen
IGNORE = {"", "Land", "Regierungsbezirk", "Statistische Kennziffer und Zulassungsbezirk"}

# Spalten, die über die Jahre umbenannt wurden → auf einheitlichen Namen normalisieren
RENAMES = {
    # FahrzeugArt: Gendering-Änderungen (2021/22 → ab 2023)
    "FZ_Krafträder_darunter weibliche Halter":                     "FZ_Krafträder_darunter Halterinnen",
    "FZ_Personenkraftwagen_und zwar gewerbliche Halter":            "FZ_Personenkraftwagen_und zwar gewerbliche Halterinnen und Halter",
    "FZ_Personenkraftwagen_und zwar weibliche Halter":              "FZ_Personenkraftwagen_und zwar Halterinnen",
    # FahrzeugArt: darunter → davon (2021 → ab 2022)
    "FZ_Zugmaschinen_darunter Sattelzugmaschinen":                  "FZ_Zugmaschinen_davon Sattelzugmaschinen",
    "FZ_Zugmaschinen_darunter land-/forstwirtschaftliche Zugmaschinen": "FZ_Zugmaschinen_davon land-/forstwirtschaftliche Zugmaschinen",
    "FZ_Zugmaschinen_land-/forstwirtschaftliche Zugmaschinen beinhalten leichte Zug-maschinen": "FZ_Zugmaschinen_davon sonstige Zugmaschinen",
    # AntriebsArt: abweichende Schreibweise der letzten Spalte
    "AA_Darunter dieselangetriebene Pkw nach Emissionsgruppen_schadstoffred. mit Dieselantrieb insgesamt":
        "AA_Darunter dieselangetriebene Pkw nach Emissionsgruppen_schadstoffreduziert mit Dieselantrieb insgesamt",
}

def bereinige(s):
    s = re.sub(r'\s+', ' ', s)  # Zeilenumbrüche normalisieren
    s = re.sub(r'- ', '', s)    # Trennstriche aus Umbrüchen entfernen
    return s.strip()

def baue_spaltennamen(zeile0, zeile1, label):
    # Oberkategorie (Zeile 0) vorwärts auffüllen
    oberkategorien = []
    letzte = ""
    for s in zeile0:
        s = bereinige(s)
        if s: letzte = s
        oberkategorien.append(letzte)

    spaltennamen = []
    for ober, unter in zip(oberkategorien, [bereinige(s) for s in zeile1]):
        if ober in IGNORE or unter in IGNORE:
            spaltennamen.append("")                          # Metadaten überspringen
        elif not ober and not unter:
            spaltennamen.append("")                          # leere Spalte
        elif not ober:
            spaltennamen.append(label + "_" + unter)
        elif not unter or ober == unter:
            spaltennamen.append(label + "_" + ober)
        else:
            spaltennamen.append(label + "_" + ober + "_" + unter)
    return spaltennamen

def lade_kiel(dateiprefix, label, jahr):
    pfad = INPUTFOLDER + str(jahr) + "/" + dateiprefix + "_" + str(jahr) + ".CSV"
    with open(pfad, encoding="latin-1", newline="") as f:
        zeilen = list(csv.reader(f, delimiter=";"))

    spaltennamen = baue_spaltennamen(zeilen[0], zeilen[1], label)

    for zeile in zeilen[2:]:
        if len(zeile) > 2 and zeile[2].strip().startswith("01002"):
            return {
                RENAMES.get(name, name): wert
                for name, wert in zip(spaltennamen, zeile)
                if name
            }

# Alle Daten laden
alle_zeilen = []
for jahr in YEARS:
    zeile = {"Jahr": jahr}
    for dateiprefix, label in SOURCES:
        zeile.update(lade_kiel(dateiprefix, label, jahr))
    alle_zeilen.append(zeile)
    print(str(jahr) + ": geladen")

# Alle Spaltennamen über alle Jahre sammeln (Reihenfolge erhalten)
alle_spalten = list(dict.fromkeys(k for z in alle_zeilen for k in z.keys()))

with open(OUTPUTFOLDER + "Kiel_Zulassungsdaten.CSV", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=alle_spalten, delimiter=",", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(alle_zeilen)

print("\nFertig! " + str(len(alle_spalten)) + " Spalten, " + str(len(alle_zeilen)) + " Zeilen gespeichert.")

2021: geladen
2022: geladen
2023: geladen
2024: geladen
2025: geladen

Fertig! 62 Spalten, 5 Zeilen gespeichert.
